# Speed Limit Detector Training

This notebook trains a YOLOv8 model against the local Roboflow-exported speed limit sign dataset bundled with the repository. It removes the original hardcoded API key dependency and trains directly from the checked-in dataset.


## Workflow

1. Resolve the dataset and repository root.
2. Start from the `yolov8n.pt` base model.
3. Train a single-class detector.
4. Optionally preview predictions with the curated weight file in `models/`.


In [ ]:
from pathlib import Path
import torch
from ultralytics import YOLO

def first_existing(*candidates: str) -> Path:
    for candidate in candidates:
        path = Path(candidate).resolve()
        if path.exists():
            return path
    raise FileNotFoundError(f"None of the candidate paths exist: {candidates}")

DATA_YAML = first_existing("data/speed_limit_signs_v1/data.yaml", "../data/speed_limit_signs_v1/data.yaml")
REPO_ROOT = DATA_YAML.parents[2]
CURATED_WEIGHTS = REPO_ROOT / "models" / "speed_limit_detector_best.pt"
DEVICE = 0 if torch.cuda.is_available() else "cpu"

print(f"Repository root: {REPO_ROOT}")
print(f"Dataset yaml: {DATA_YAML}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Curated weights present: {CURATED_WEIGHTS.exists()}")


In [ ]:
model = YOLO("yolov8n.pt")

train_results = model.train(
    data=str(DATA_YAML),
    epochs=10,
    imgsz=640,
    device=DEVICE,
    project=str(REPO_ROOT / "runs"),
    name="portfolio_speed_limit_detector",
    exist_ok=True,
)

train_results


In [ ]:
preview_model_path = CURATED_WEIGHTS if CURATED_WEIGHTS.exists() else Path("yolov8n.pt")
preview_model = YOLO(str(preview_model_path))
sample_images = sorted((DATA_YAML.parent / "test" / "images").glob("*.jpg"))[:3]

preview_results = preview_model.predict(
    source=[str(path) for path in sample_images],
    conf=0.25,
    project=str(REPO_ROOT / "runs"),
    name="portfolio_preview",
    exist_ok=True,
    save=True,
)

preview_results
